In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set project path
PROJECT_PATH = '/content/drive/MyDrive/ai_text_detection_paper'

# Change working directory
import os
os.chdir(PROJECT_PATH)
print(f"✅ Working directory: {os.getcwd()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Working directory: /content/drive/MyDrive/ai_text_detection_paper


In [ ]:
!pip install -q datasets pandas scikit-learn

import pandas as pd
import numpy as np
from datasets import load_dataset
import re
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


In [ ]:
# Check what files are in data/raw/
raw_data_path = os.path.join(PROJECT_PATH, 'data/raw')

print("📂 Files in data/raw/:")
for file in os.listdir(raw_data_path):
    file_path = os.path.join(raw_data_path, file)
    if os.path.isfile(file_path):
        size_mb = os.path.getsize(file_path) / (1024 * 1024)
        print(f"   • {file} ({size_mb:.2f} MB)")

📂 Files in data/raw/:
   • train_v2_drcat_02.csv (97.19 MB)
   • train.csv (153.13 MB)
   • all.jsonl (70.31 MB)


In [ ]:
# Load HC3 train.csv
hc3_path = os.path.join(PROJECT_PATH, 'data/raw/train.csv')

print("📥 Loading HC3 train.csv...")
hc3_df = pd.read_csv(hc3_path)

print(f"✅ HC3 loaded!")
print(f"   Shape: {hc3_df.shape}")
print(f"   Columns: {list(hc3_df.columns)}")
print(f"\n📋 First 3 rows:")
print(hc3_df.head(3))
print(f"\n📊 Data types:")
print(hc3_df.dtypes)

📥 Loading HC3 train.csv...
✅ HC3 loaded!
   Shape: (7321, 8)
   Columns: ['prompt', 'Human_story', 'gemma-2-9b', 'mistral-7B', 'qwen-2-72B', 'llama-8B', 'accounts/yi-01-ai/models/yi-large', 'GPT_4-o']

📋 First 3 rows:
                                              prompt  \
0                  The Human Toll of Nuclear Testing   
1  In the age of coronavirus, the only way you ca...   
2  Roberta Karmel, First Woman Named to the S.E.C...   

                                         Human_story  \
0  Comments\nThe U.S. bombings thatended World Wa...   
1  new video loaded:Messages From Quarantine\ntra...   
2  Supported by\nRoberta Karmel, First Woman Name...   

                                          gemma-2-9b  \
0  ## The Unseen Scars: The Enduring Human Toll o...   
1  ## In the Age of Coronavirus, the Only Way You...   
2  ## Roberta Karmel, First Woman Named to the S....   

                                          mistral-7B  \
0  \n\nTitle: The Atomic Aftermath: The U.S. Bomb..

In [ ]:
# Load gsingh train.csv
gsingh_path = os.path.join(PROJECT_PATH, 'data/raw/all.jsonl')

print("📥 Loading gsingh all.jsonl...")
gsingh_df = pd.read_json(gsingh_path, lines = True)

print(f"✅ gsingh loaded!")
print(f"   Shape: {gsingh_df.shape}")
print(f"   Columns: {list(gsingh_df.columns)}")
print(f"\n📋 First 3 rows:")
print(gsingh_df.head(3))
print(f"\n📊 Data types:")
print(gsingh_df.dtypes)

📥 Loading gsingh all.jsonl...
✅ gsingh loaded!
   Shape: (24322, 5)
   Columns: ['question', 'human_answers', 'chatgpt_answers', 'index', 'source']

📋 First 3 rows:
                                            question  \
0  Why is every book I hear about a " NY Times # ...   
1  If salt is so bad for cars , why do we use it ...   
2  Why do we still have SD TV channels when HD lo...   

                                       human_answers  \
0  [Basically there are many categories of " Best...   
1  [salt is good for not dying in car crashes and...   
2  [The way it works is that old TV stations got ...   

                                     chatgpt_answers  index       source  
0  [There are many different best seller lists th...    NaN  reddit_eli5  
1  [Salt is used on roads to help melt ice and sn...    NaN  reddit_eli5  
2  [There are a few reasons why we still have SD ...    NaN  reddit_eli5  

📊 Data types:
question            object
human_answers       object
chatgpt_answers 

In [ ]:
# Check HC3 structure - looks clean already
print("=" * 60)
print("HC3 DATASET STRUCTURE")
print("=" * 60)
print(f"Total rows: {len(hc3_df)}")
print(f"\n✅ HC3 has clean text columns (no bracket artifacts)")
print(f"   • Human_story: {type(hc3_df['Human_story'].iloc[0])}")
print(f"   • AI columns: 6 different models")

# Check gsingh structure - has list-like strings
print("\n" + "=" * 60)
print("GSINGH DATASET STRUCTURE")
print("=" * 60)
print(f"Total rows: {len(gsingh_df)}")
print(f"\n⚠️ gsingh has list-like strings with brackets:")
print(f"\nSample human_answers (first 200 chars):")
print(gsingh_df['human_answers'].iloc[0][:200])
print(f"\nType: {type(gsingh_df['human_answers'].iloc[0])}")

print(f"\n\nSample chatgpt_answers (first 200 chars):")
print(gsingh_df['chatgpt_answers'].iloc[0][:200])

HC3 DATASET STRUCTURE
Total rows: 7321

✅ HC3 has clean text columns (no bracket artifacts)
   • Human_story: <class 'str'>
   • AI columns: 6 different models

GSINGH DATASET STRUCTURE
Total rows: 24322

⚠️ gsingh has list-like strings with brackets:

Sample human_answers (first 200 chars):
['Basically there are many categories of " Best Seller " . Replace " Best Seller " by something like " Oscars " and every " best seller " book is basically an " oscar - winning " book . May not have won the " Best film " , but even if you won the best director or best script , you \'re still an " oscar - winning " film . Same thing for best sellers . Also , IIRC the rankings change every week or something like that . Some you might not be best seller one week , but you may be the next week . I guess even if you do n\'t stay there for long , you still achieved the status . Hence , # 1 best seller .', "If you 're hearing about it , it 's because it was a very good or very well - publicized book ( or 

In [ ]:
def extract_text_from_list(text_field):
    """
    Extract text from list-like fields in gsingh dataset.
    Takes first element if list, returns as-is if string.
    """
    if isinstance(text_field, list):
        if len(text_field) > 0:
            return text_field[0]  # Take first answer
        else:
            return ""
    elif isinstance(text_field, str):
        return text_field
    else:
        return ""

# Test the function
test_sample = gsingh_df['human_answers'].iloc[0]
print("📝 Original (list):")
print(f"Type: {type(test_sample)}")
print(f"Length: {len(test_sample)} items")
print(f"\n✅ Extracted (string):")
extracted = extract_text_from_list(test_sample)
print(f"Type: {type(extracted)}")
print(f"First 200 chars: {extracted[:200]}")

📝 Original (list):
Type: <class 'list'>
Length: 3 items

✅ Extracted (string):
Type: <class 'str'>
First 200 chars: Basically there are many categories of " Best Seller " . Replace " Best Seller " by something like " Oscars " and every " best seller " book is basically an " oscar - winning " book . May not have won


In [ ]:
def clean_text(text):
    """
    Clean text by:
    - Removing extra whitespace
    - Removing newlines
    - Stripping leading/trailing spaces
    """
    if pd.isna(text) or text == "":
        return ""

    # Convert to string
    text = str(text)

    # Remove multiple spaces
    text = re.sub(r'\s+', ' ', text)

    # Strip
    text = text.strip()

    return text

# Test the function
test_text = "  This   is\n\na    test  \n  string.  "
print(f"Original: '{test_text}'")
print(f"Cleaned: '{clean_text(test_text)}'")

Original: '  This   is

a    test  
  string.  '
Cleaned: 'This is a test string.'


In [ ]:
print("🔄 Processing HC3 dataset...")

hc3_rows = []

for idx, row in hc3_df.iterrows():
    # Human text
    human_text = clean_text(row['Human_story'])
    if len(human_text) >= 30:  # Filter short texts
        hc3_rows.append({
            'text': human_text,
            'label': 0,  # Human
            'source': 'hc3'
        })

    # AI texts from all model columns
    ai_columns = ['gemma-2-9b', 'mistral-7B', 'qwen-2-72B',
                  'llama-8B', 'accounts/yi-01-ai/models/yi-large', 'GPT_4-o']

    for col in ai_columns:
        ai_text = clean_text(row[col])
        if len(ai_text) >= 30:  # Filter short texts
            hc3_rows.append({
                'text': ai_text,
                'label': 1,  # AI
                'source': f'hc3_{col}'
            })

hc3_processed = pd.DataFrame(hc3_rows)

print(f"✅ HC3 processed!")
print(f"   Total samples: {len(hc3_processed)}")
print(f"   Human (label=0): {(hc3_processed['label'] == 0).sum()}")
print(f"   AI (label=1): {(hc3_processed['label'] == 1).sum()}")
print(f"\n📋 Sample:")
print(hc3_processed.head(3))

🔄 Processing HC3 dataset...
✅ HC3 processed!
   Total samples: 51179
   Human (label=0): 7293
   AI (label=1): 43886

📋 Sample:
                                                text  label          source
0  Comments The U.S. bombings thatended World War...      0             hc3
1  ## The Unseen Scars: The Enduring Human Toll o...      1  hc3_gemma-2-9b
2  Title: The Atomic Aftermath: The U.S. Bombings...      1  hc3_mistral-7B


In [ ]:
print("🔄 Processing gsingh dataset...")

gsingh_rows = []

for idx, row in gsingh_df.iterrows():
    # Extract and clean human answers
    human_text = extract_text_from_list(row['human_answers'])
    human_text = clean_text(human_text)

    if len(human_text) >= 30:  # Filter short texts
        gsingh_rows.append({
            'text': human_text,
            'label': 0,  # Human
            'source': 'gsingh'
        })

    # Extract and clean AI answers
    ai_text = extract_text_from_list(row['chatgpt_answers'])
    ai_text = clean_text(ai_text)

    if len(ai_text) >= 30:  # Filter short texts
        gsingh_rows.append({
            'text': ai_text,
            'label': 1,  # AI
            'source': 'gsingh'
        })

gsingh_processed = pd.DataFrame(gsingh_rows)

print(f"✅ gsingh processed!")
print(f"   Total samples: {len(gsingh_processed)}")
print(f"   Human (label=0): {(gsingh_processed['label'] == 0).sum()}")
print(f"   AI (label=1): {(gsingh_processed['label'] == 1).sum()}")
print(f"\n📋 Sample:")
print(gsingh_processed.head(3))

🔄 Processing gsingh dataset...
✅ gsingh processed!
   Total samples: 48162
   Human (label=0): 24302
   AI (label=1): 23860

📋 Sample:
                                                text  label  source
0  Basically there are many categories of " Best ...      0  gsingh
1  There are many different best seller lists tha...      1  gsingh
2  salt is good for not dying in car crashes and ...      0  gsingh


In [ ]:
# Load drcat dataset
drcat_path = '/content/drive/MyDrive/ai_text_detection_paper/data/raw/train_v2_drcat_02.csv'

print(f"📥 Loading train_v2_drcat_02.csv...")
drcat_df = pd.read_csv(drcat_path)

print(f"✅ Loaded!")
print(f"   Shape: {drcat_df.shape}")
print(f"   Columns: {list(drcat_df.columns)}")

print(f"\n📋 First 3 rows:")
print(drcat_df.head(3))

print(f"\n📊 Data types:")
print(drcat_df.dtypes)

# Check for label column
if 'label' in drcat_df.columns:
    print(f"\n📊 Label distribution:")
    print(drcat_df['label'].value_counts())
elif 'source' in drcat_df.columns:
    print(f"\n📊 Source distribution:")
    print(drcat_df['source'].value_counts())

# Check text length statistics
if 'text' in drcat_df.columns:
    print(f"\n📏 Text length statistics:")
    drcat_df['text_length'] = drcat_df['text'].astype(str).str.len()
    print(drcat_df['text_length'].describe())

📥 Loading train_v2_drcat_02.csv...
✅ Loaded!
   Shape: (44868, 5)
   Columns: ['text', 'label', 'prompt_name', 'source', 'RDizzl3_seven']

📋 First 3 rows:
                                                text  label  \
0  Phones\n\nModern humans today are always on th...      0   
1  This essay will explain if drivers should or s...      0   
2  Driving while the use of cellular devices\n\nT...      0   

          prompt_name           source  RDizzl3_seven  
0  Phones and driving  persuade_corpus          False  
1  Phones and driving  persuade_corpus          False  
2  Phones and driving  persuade_corpus          False  

📊 Data types:
text             object
label             int64
prompt_name      object
source           object
RDizzl3_seven      bool
dtype: object

📊 Label distribution:
label
0    27371
1    17497
Name: count, dtype: int64

📏 Text length statistics:
count    44868.000000
mean      2216.222921
std        969.928064
min         48.000000
25%       1564.750000
50%  

In [ ]:
print("🔍 Analyzing prompts/topics across datasets...\n")

# HC3 - check if it has prompt column
print("=" * 60)
print("HC3 DATASET")
print("=" * 60)
if 'prompt' in hc3_df.columns:
    print(f"✅ Has 'prompt' column")
    print(f"   Unique prompts: {hc3_df['prompt'].nunique()}")
    print(f"\n   Sample prompts:")
    print(hc3_df['prompt'].value_counts().head(10))
else:
    print("❌ No prompt column")

# gsingh - check question column
print("\n" + "=" * 60)
print("GSINGH DATASET")
print("=" * 60)
if 'question' in gsingh_df.columns:
    print(f"✅ Has 'question' column")
    print(f"   Unique questions: {gsingh_df['question'].nunique()}")
    print(f"\n   Sample questions:")
    print(gsingh_df['question'].value_counts().head(10))
else:
    print("❌ No question column")

# drcat - check prompt_name column
print("\n" + "=" * 60)
print("DRCAT DATASET")
print("=" * 60)
if 'prompt_name' in drcat_df.columns:
    print(f"✅ Has 'prompt_name' column")
    print(f"   Unique prompts: {drcat_df['prompt_name'].nunique()}")
    print(f"\n   Sample prompts:")
    print(drcat_df['prompt_name'].value_counts().head(10))
else:
    print("❌ No prompt column")

🔍 Analyzing prompts/topics across datasets...

HC3 DATASET
✅ Has 'prompt' column
   Unique prompts: 6722

   Sample prompts:
prompt
Here’s what you need to know at the end of the day.                                                        43
Test your knowledge of this week’s health news.                                                            32
The label shows its spring looks.                                                                          22
See full results and maps from the California election.                                                    19
The fall 2018 men’s collection.                                                                            16
Did you follow the news this week? Take our quiz to see how well you stack up with other Times readers.    13
How different groups voted                                                                                 13
Photos from The New York Times and photographers from around the world.                           

In [ ]:
print("🔄 Processing drcat dataset with prompt tracking...\n")

# Process drcat
drcat_rows_with_prompts = []

for idx, row in drcat_df.iterrows():
    text = clean_text(row['text'])

    if len(text) >= 30:  # Filter short texts
        drcat_rows_with_prompts.append({
            'text': text,
            'label': row['label'],
            'source': 'drcat',
            'prompt_id': f"drcat_{row['prompt_name']}",
            'prompt_text': row['prompt_name']
        })

drcat_processed = pd.DataFrame(drcat_rows_with_prompts)

print("✅ drcat processed and prompts tracked")
print(f"   Total samples: {len(drcat_processed)}")
print(f"   Human (label=0): {(drcat_processed['label'] == 0).sum()}")
print(f"   AI (label=1): {(drcat_processed['label'] == 1).sum()}")
print(f"   Unique prompts: {drcat_processed['prompt_id'].nunique()}")

print("\n" + "=" * 60)
print("📊 SUMMARY - ALL DATASETS WITH PROMPTS")
print("=" * 60)
print(f"HC3:")
print(f"   Samples: {len(hc3_processed)}")
print(f"   Unique prompts: {hc3_processed['prompt_id'].nunique()}")

print(f"\ngsingh:")
print(f"   Samples: {len(gsingh_processed)}")
print(f"   Unique prompts: {gsingh_processed['prompt_id'].nunique()}")

print(f"\ndrcat:")
print(f"   Samples: {len(drcat_processed)}")
print(f"   Unique prompts: {drcat_processed['prompt_id'].nunique()}")

print(f"\nTotal unique prompts: {hc3_processed['prompt_id'].nunique() + gsingh_processed['prompt_id'].nunique() + drcat_processed['prompt_id'].nunique()}")

🔄 Processing drcat dataset with prompt tracking...

✅ drcat processed and prompts tracked
   Total samples: 44868
   Human (label=0): 27371
   AI (label=1): 17497
   Unique prompts: 15

📊 SUMMARY - ALL DATASETS WITH PROMPTS
HC3:
   Samples: 51179
   Unique prompts: 6713

gsingh:
   Samples: 48162
   Unique prompts: 24322

drcat:
   Samples: 44868
   Unique prompts: 15

Total unique prompts: 31050


In [ ]:
print("🔗 Combining all datasets...\n")

# Combine all three datasets
all_data = pd.concat([hc3_processed, gsingh_processed, drcat_processed], ignore_index=True)

print("✅ All datasets combined!")
print(f"   Total samples: {len(all_data):,}")
print(f"   Human (label=0): {(all_data['label'] == 0).sum():,}")
print(f"   AI (label=1): {(all_data['label'] == 1).sum():,}")
print(f"   Unique prompts: {all_data['prompt_id'].nunique():,}")

print(f"\n📊 Class distribution:")
print(all_data['label'].value_counts())
print(f"\n   Imbalance ratio (Human:AI): 1:{(all_data['label'] == 1).sum() / (all_data['label'] == 0).sum():.2f}")

print(f"\n📊 Source distribution:")
print(all_data['source'].value_counts())

print(f"\n📋 Sample of combined data:")
print(all_data.head(3))

🔗 Combining all datasets...

✅ All datasets combined!
   Total samples: 144,209
   Human (label=0): 58,966
   AI (label=1): 85,243
   Unique prompts: 31,050

📊 Class distribution:
label
1    85243
0    58966
Name: count, dtype: int64

   Imbalance ratio (Human:AI): 1:1.45

📊 Source distribution:
source
gsingh                                   48162
drcat                                    44868
hc3_GPT_4-o                               7321
hc3_accounts/yi-01-ai/models/yi-large     7319
hc3_mistral-7B                            7316
hc3_qwen-2-72B                            7314
hc3_gemma-2-9b                            7310
hc3_llama-8B                              7306
hc3                                       7293
Name: count, dtype: int64

📋 Sample of combined data:
                                                text  label          source  \
0  Comments The U.S. bombings thatended World War...      0             hc3   
1  ## The Unseen Scars: The Enduring Human Toll o...      1  

In [ ]:
print("🔍 Checking for duplicate texts...\n")

print(f"Before removing duplicates: {len(all_data):,} samples")

# Remove duplicate texts (keep first occurrence)
all_data_dedup = all_data.drop_duplicates(subset=['text'], keep='first').copy()

print(f"After removing duplicates: {len(all_data_dedup):,} samples")
print(f"Duplicates removed: {len(all_data) - len(all_data_dedup):,}")

print(f"\n📊 Class distribution after deduplication:")
print(all_data_dedup['label'].value_counts())
print(f"\n   Imbalance ratio (Human:AI): 1:{(all_data_dedup['label'] == 1).sum() / (all_data_dedup['label'] == 0).sum():.2f}")

# Update the dataframe
all_data = all_data_dedup.reset_index(drop=True)

🔍 Checking for duplicate texts...

Before removing duplicates: 144,209 samples
After removing duplicates: 141,656 samples
Duplicates removed: 2,553

📊 Class distribution after deduplication:
label
1    84633
0    57023
Name: count, dtype: int64

   Imbalance ratio (Human:AI): 1:1.48


In [ ]:
print("⚖️ Balancing the dataset...\n")

from sklearn.utils import resample

# Separate by class
human_data = all_data[all_data['label'] == 0].copy()
ai_data = all_data[all_data['label'] == 1].copy()

print(f"Before balancing:")
print(f"   Human: {len(human_data):,}")
print(f"   AI: {len(ai_data):,}")

# Undersample AI to match human count
ai_balanced = resample(ai_data,
                       n_samples=len(human_data),
                       random_state=42,
                       replace=False)

# Combine balanced data
balanced_data = pd.concat([human_data, ai_balanced], ignore_index=True)

# Shuffle
balanced_data = balanced_data.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\n✅ Dataset balanced!")
print(f"   Total samples: {len(balanced_data):,}")
print(f"   Human (label=0): {(balanced_data['label'] == 0).sum():,}")
print(f"   AI (label=1): {(balanced_data['label'] == 1).sum():,}")
print(f"   Balance ratio: 1:{(balanced_data['label'] == 1).sum() / (balanced_data['label'] == 0).sum():.2f}")

print(f"\n📊 Source distribution in balanced data:")
print(balanced_data['source'].value_counts())

⚖️ Balancing the dataset...

Before balancing:
   Human: 57,023
   AI: 84,633

✅ Dataset balanced!
   Total samples: 114,046
   Human (label=0): 57,023
   AI (label=1): 57,023
   Balance ratio: 1:1.00

📊 Source distribution in balanced data:
source
drcat                                    39108
gsingh                                   38119
hc3                                       7233
hc3_mistral-7B                            4984
hc3_qwen-2-72B                            4960
hc3_GPT_4-o                               4946
hc3_llama-8B                              4928
hc3_gemma-2-9b                            4903
hc3_accounts/yi-01-ai/models/yi-large     4865
Name: count, dtype: int64


In [ ]:
print("🔀 Creating prompt-based train/val/test splits...\n")

from sklearn.model_selection import train_test_split

# Get unique prompts
unique_prompts = balanced_data['prompt_id'].unique()
print(f"Total unique prompts: {len(unique_prompts):,}")

# Split prompts (not samples) into train/val/test: 70/15/15
train_prompts, temp_prompts = train_test_split(
    unique_prompts, test_size=0.30, random_state=42
)

val_prompts, test_prompts = train_test_split(
    temp_prompts, test_size=0.50, random_state=42
)

print(f"\nPrompt splits:")
print(f"   Train prompts: {len(train_prompts):,} (70%)")
print(f"   Val prompts: {len(val_prompts):,} (15%)")
print(f"   Test prompts: {len(test_prompts):,} (15%)")

# Create splits based on prompts
train_data = balanced_data[balanced_data['prompt_id'].isin(train_prompts)].copy()
val_data = balanced_data[balanced_data['prompt_id'].isin(val_prompts)].copy()
test_data = balanced_data[balanced_data['prompt_id'].isin(test_prompts)].copy()

print(f"\n✅ Data splits created!")
print(f"\nSample counts:")
print(f"   Train: {len(train_data):,} samples")
print(f"   Val: {len(val_data):,} samples")
print(f"   Test: {len(test_data):,} samples")

print(f"\nClass distribution in each split:")
print(f"\nTrain:")
print(train_data['label'].value_counts())
print(f"   Balance: {(train_data['label'] == 0).sum()} human, {(train_data['label'] == 1).sum()} AI")

print(f"\nVal:")
print(val_data['label'].value_counts())
print(f"   Balance: {(val_data['label'] == 0).sum()} human, {(val_data['label'] == 1).sum()} AI")

print(f"\nTest:")
print(test_data['label'].value_counts())
print(f"   Balance: {(test_data['label'] == 0).sum()} human, {(test_data['label'] == 1).sum()} AI")

# Verify no prompt overlap
print(f"\n🔍 Verification - No prompt leakage:")
print(f"   Train ∩ Val: {len(set(train_prompts) & set(val_prompts))} prompts")
print(f"   Train ∩ Test: {len(set(train_prompts) & set(test_prompts))} prompts")
print(f"   Val ∩ Test: {len(set(val_prompts) & set(test_prompts))} prompts")

🔀 Creating prompt-based train/val/test splits...

Total unique prompts: 30,072

Prompt splits:
   Train prompts: 21,050 (70%)
   Val prompts: 4,511 (15%)
   Test prompts: 4,511 (15%)

✅ Data splits created!

Sample counts:
   Train: 78,738 samples
   Val: 18,590 samples
   Test: 16,718 samples

Class distribution in each split:

Train:
label
1    39501
0    39237
Name: count, dtype: int64
   Balance: 39237 human, 39501 AI

Val:
label
1    9831
0    8759
Name: count, dtype: int64
   Balance: 8759 human, 9831 AI

Test:
label
0    9027
1    7691
Name: count, dtype: int64
   Balance: 9027 human, 7691 AI

🔍 Verification - No prompt leakage:
   Train ∩ Val: 0 prompts
   Train ∩ Test: 0 prompts
   Val ∩ Test: 0 prompts


In [ ]:
print("💾 Saving preprocessed data...\n")

# Create splits directory if it doesn't exist
splits_dir = os.path.join(PROJECT_PATH, 'data/splits')
os.makedirs(splits_dir, exist_ok=True)

# Save train/val/test splits
train_path = os.path.join(splits_dir, 'train.csv')
val_path = os.path.join(splits_dir, 'val.csv')
test_path = os.path.join(splits_dir, 'test.csv')

train_data.to_csv(train_path, index=False)
val_data.to_csv(val_path, index=False)
test_data.to_csv(test_path, index=False)

print("✅ Splits saved!")
print(f"   📁 {train_path}")
print(f"   📁 {val_path}")
print(f"   📁 {test_path}")

# Also save the full balanced dataset
balanced_path = os.path.join(PROJECT_PATH, 'data/preprocessed.csv')
balanced_data.to_csv(balanced_path, index=False)

print(f"\n✅ Full balanced dataset saved!")
print(f"   📁 {balanced_path}")

# Save dataset statistics (convert numpy types to Python types)
stats = {
    'total_samples': int(len(balanced_data)),
    'train_samples': int(len(train_data)),
    'val_samples': int(len(val_data)),
    'test_samples': int(len(test_data)),
    'unique_prompts': int(len(unique_prompts)),
    'train_prompts': int(len(train_prompts)),
    'val_prompts': int(len(val_prompts)),
    'test_prompts': int(len(test_prompts)),
    'human_samples': int((balanced_data['label'] == 0).sum()),
    'ai_samples': int((balanced_data['label'] == 1).sum())
}

import json
stats_path = os.path.join(PROJECT_PATH, 'data/preprocessing_stats.json')
with open(stats_path, 'w') as f:
    json.dump(stats, f, indent=4)

print(f"\n✅ Statistics saved!")
print(f"   📁 {stats_path}")

print("\n" + "=" * 60)
print("📊 FINAL PREPROCESSING SUMMARY")
print("=" * 60)
for key, value in stats.items():
    print(f"   {key}: {value:,}")

💾 Saving preprocessed data...

✅ Splits saved!
   📁 /content/drive/MyDrive/ai_text_detection_paper/data/splits/train.csv
   📁 /content/drive/MyDrive/ai_text_detection_paper/data/splits/val.csv
   📁 /content/drive/MyDrive/ai_text_detection_paper/data/splits/test.csv

✅ Full balanced dataset saved!
   📁 /content/drive/MyDrive/ai_text_detection_paper/data/preprocessed.csv

✅ Statistics saved!
   📁 /content/drive/MyDrive/ai_text_detection_paper/data/preprocessing_stats.json

📊 FINAL PREPROCESSING SUMMARY
   total_samples: 114,046
   train_samples: 78,738
   val_samples: 18,590
   test_samples: 16,718
   unique_prompts: 30,072
   train_prompts: 21,050
   val_prompts: 4,511
   test_prompts: 4,511
   human_samples: 57,023
   ai_samples: 57,023


In [ ]:
print("📋 Sample of preprocessed data:\n")

print("=" * 60)
print("TRAIN SET SAMPLE")
print("=" * 60)
print(train_data[['text', 'label', 'source', 'prompt_text']].head(3))

print("\n" + "=" * 60)
print("VALIDATION SET SAMPLE")
print("=" * 60)
print(val_data[['text', 'label', 'source', 'prompt_text']].head(3))

print("\n" + "=" * 60)
print("TEST SET SAMPLE")
print("=" * 60)
print(test_data[['text', 'label', 'source', 'prompt_text']].head(3))

print("\n" + "=" * 60)
print("✅ PREPROCESSING COMPLETE!")
print("=" * 60)
print("\n📁 Files created in Google Drive:")
print("   • data/preprocessed.csv (114,046 samples)")
print("   • data/splits/train.csv (78,738 samples)")
print("   • data/splits/val.csv (18,590 samples)")
print("   • data/splits/test.csv (16,718 samples)")
print("   • data/preprocessing_stats.json")

print("\n🎯 Key Achievements:")
print("   ✅ Perfect class balance (1:1 ratio)")
print("   ✅ No data leakage (0 prompt overlap between splits)")
print("   ✅ Clean text (duplicates removed, minimum length enforced)")
print("   ✅ Prompt tracking for reproducibility")
print("   ✅ Research-ready dataset")

📋 Sample of preprocessed data:

TRAIN SET SAMPLE
                                                text  label  source  \
0  No . First , 1 ly is around 10 ^ 15 meters , s...      0  gsingh   
1  DEAR PRINCIPLE I think that we shouldn't have ...      0   drcat   
2  While COBRA premiums are not eligible to be a ...      0  gsingh   

                                         prompt_text  
0  If I had a stick 1 lightyear long and I moved ...  
1              Grades for extracurricular activities  
2             New 1099 employee with Cobra insurance  

VALIDATION SET SAMPLE
                                                text  label source  \
3  The New York Times Business|Growth and Governm...      0    hc3   
6  In my opinion, distance learning is a great op...      1  drcat   
7  Covid-19Guidance Contra Costa County, Californ...      0    hc3   

                                         prompt_text  
3  As a gift this summer, Donald J. Trump sent Sh...  
6                               